<a href="https://colab.research.google.com/github/snr-laboratory/snrlab-ic-q-pix-v1/blob/main/dev_journals/kgosine_202511_a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### LVDS Receiver Again

![](https://drive.google.com/uc?export=view&id=1SbMEYKqQ0lDHINUuJhA0yfMvI4l0iA7L)


Note that the receiver needs 5uA applied externally. Need to figure out the equivalent of that in voltage across res.

Post-layout transient simulation:
![](https://drive.google.com/uc?export=view&id=1qBoS2SOC22osCuzOF2q3xCiQpCfclBQY)

48uW average power consumption with a 5uA bias current.
To determine if this is an acceptible rise time for the circuit, it is tested with the clocked comparator. The clocekd comparator also requuires a clkbar signal so the clk (lvds rx output) must go both into the comparator and an inverter.

![](https://drive.google.com/uc?export=view&id=1Sm_oQeXRGz1mh7gKfaT6oadFJOeakCMq)


![](https://drive.google.com/uc?export=view&id=1V2CDv1flWSxYaNHKSqBrGmGR9K1nUU4h)

The RX output in this context is slightly worse, indicating that the load on the receiver is actually more than 20fF. The inverter uses 2.4uW of average power and is able to recover a good ClkBar signal from the Clk signal. Regardless, the comparator seems fairly robust to the rise time of the RX output (clock). The difference is most notable in the on-time of the Comparator Output (Q) which are the calls for replenishment. Varying the bias current of the RX from 2uA to 5uA will vary the on-time between 9ns-11ns. In testing, the fidelity of the clk signal is best judged by looking at CLKBAR rather than CLK since even a long rise time in Clk can be successful in the comparator. If we had more power to spare we could increase the drive strength of the reciever but for now this is fine.

## Removing Buffering from Comparator Block

Removed the buffering & load transistors on the comparator to be placed in a separate block. The comparator output inverter & buffer circuit was sized to be low power which was overlooked previously. The average power consumption of the comparator circuit is 10.99uW when the input is less than threshold (no replenishment call) and 10.92uW whenthe input is above threshold.
Including the power consumption from the comparator, the buffering circuit, and the inverter whic converts an ideal clk to clkbar the average power consumption is 17.71uW. This is pre-layout simulation.

![](https://drive.google.com/uc?export=view&id=1UE-njTwwu52o1cR8WK8RQOJbVO4r9IAE)

![](https://drive.google.com/uc?export=view&id=19GFseJxDh0MU86KutQiqq2wYTuwbackr)



Post layout simulation calls for an additional 4.5fC on OUTP to balance the loads on the comparator outputs, this needs to be adjusted into the transistor sizes. The power consumption fo the comparator is now 13uW and for the buffer circuit, 5.2uW.


## Post Layout Simulation of QPix Loop with LVDS RX & Sized Buffers/Inverters

![](https://drive.google.com/uc?export=view&id=1ix8_-cIRJ62kFKpBAKRKcYMXNiidZpRX)

No 3pF input cap. Post layout on all components. First with the 2fC input signal:

![](https://drive.google.com/uc?export=view&id=177_mS3rPVNAE904U9_V2jzKcuwAZ32Hf)

![](https://drive.google.com/uc?export=view&id=1ekpKhM5Oc3lktUCL7Fj_XCnC_9dSTzRU)



And with a realistic charge input:

![](https://drive.google.com/uc?export=view&id=1g4HZg_D1Q9j1PyDZ37EQs03wV1c7dPA2)

![](https://drive.google.com/uc?export=view&id=1duw8vT_UUCZoDa6v7pW1S42j1PYn5KIs)

![](https://drive.google.com/uc?export=view&id=1i4vTe6nbhXo9oAYWYKK57IU6D25KlYFD)


Average power consumption
OpAmp: 36.09uW
Comparator: 12.83uW
Comparator buffer: 0.4793 uW
Replenishment switch: 978.8pW
LVDS RX: 44.99uW
Clk inverter: 2.887uW

For the analog components, the sum is 49.4uW.

### POST layout simulation of all components with dummies

![](https://drive.google.com/uc?export=view&id=1q3ff3tXmBC9eZDz7Nc4avaeIbEcMhCtM)

![](https://drive.google.com/uc?export=view&id=1cfVsOaTQw34vvDKaDcksbHTFQ0i1ZWxc)

![](https://drive.google.com/uc?export=view&id=1psc8HSyC40CKg1qymu2uB8dM8YVeAYO2)


### Standard buffer/inverter for output transmission

![](https://drive.google.com/uc?export=view&id=1f17l64xMURjPV26Bwi_NEQ1Dmkmei8Tk)

The buffer/inverter has a power consumption of 665uW/662uW when transmitting constant (every 10ns) pulses. When there are pulses to be transmitted, the power consumption for the buffer/inverter in 128pW/68pW.


### CLK spy point

To confirm that the input clock has been recovered properly by the LVDS RX, there should be a spy point at the CLKBAR output. Because of the high frequency of this signal to be recovered, a source follower spy point like that used in the integrator output won't be effective so I used the same standard buffer to get this signal out. The power consumption of this block is 500uW. In order to only draw power when we want to check the spy point, I suggest that the VDD of the buffer be connected to a pad and supplied externally. That way the block only consumes power when we apply VDD and wish to check that the clock was recovered.

![](https://drive.google.com/uc?export=view&id=1wzMahtAyJZxL5QQwFvbrZNfZtg-uhnkm)
